In [1]:
print(123)

123


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [6]:
def llm(prompt):
    response = openai_client.responses.create(
        model="openai/gpt-oss-120b",
        input=prompt
    )
    return response.output_text

In [5]:
llm("Hey, what's up?")


'Hey there! Not much—just here and ready to help out. How’s your day going? Anything fun or interesting on your mind today?'

In [7]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Absolutely! Most of our courses are open enrollment, so you can sign up at any time. Here’s a quick rundown of what you need to do:

1. **Find the course** – Go to the course catalog on our website and locate the program you’re interested in. If you need help finding it, just let me know the name or subject and I can point you in the right direction.

2. **Check any prerequisites** – Some courses have recommended background knowledge (e.g., basic programming, statistics, etc.). If there are any, they’ll be listed on the course page. Don’t worry—most introductory courses have no formal prerequisites.

3. **Create or log in to your account** – You’ll need an account to enroll. If you don’t have one yet, click **Sign Up** and follow the prompts (just an email and a password).

4. **Enroll** – Once you’re logged in, click the **Enroll Now** button on the course page. You’ll be taken through the payment (if it’s a paid course) or confirmation steps (for free courses).

5. **Start learning**

In [8]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [9]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [11]:
answer = llm(prompt)
print(answer)

Yes—you can still join the course. Just keep in mind that if you want to earn a certificate, you’ll need to submit your project before the submission deadline.


In [12]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [15]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

print(courses_raw)

[{'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 472}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 79}, {'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 402}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 255}]


In [16]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1208

In [17]:
documents[500]

{'id': '4d8c1f5b3a',
 'course': 'llm-zoomcamp',
 'section': 'Module 1: Introduction to LLMs and RAG',
 'question': 'How do I count tokens for a non-OpenAI model (Gemini, Mistral, HuggingFace)?',
 'answer': '`tiktoken` only ships tokenizers for OpenAI models. Using `cl100k_base` for other providers gives wrong counts and unreliable cost estimates.\n\nFor other providers, use their native tokenizer:\n\n- Gemini:\n  ```python\n  import google.generativeai as genai\n  model = genai.GenerativeModel(\'gemini-2.0-flash\')\n  print(model.count_tokens(prompt))\n  ```\n- Hugging Face / open-source models:\n  ```python\n  from transformers import AutoTokenizer\n  tok = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B")\n  print(len(tok.encode(prompt)))\n  ```\n- Mistral: use the official `mistral-common` tokenizer package.\n\nDon\'t use `cl100k_base` as a generic fallback — token counts will diverge from what the provider actually bills.'}

In [18]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [19]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [20]:
[doc["question"] for doc in search_results]


['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'When will the course be offered next?',
 'I missed the first homework - can I still get a certificate?']

In [21]:
results = index.search(
    question,
    num_results=5,
    boost_dict={"question": 2.0, "section": 0.5}
)

In [22]:
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)

In [23]:
[doc["question"] for doc in results]


['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?']

In [24]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [25]:
search_results = search(question)


In [26]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c